In [2]:
import sys
import os
import time
import uuid
current_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(current_dir, ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Setup context
from dotenv import load_dotenv
load_dotenv()
from core import enable_logging
enable_logging()
# 将项目根目录加入模块路径
from agent.BasicAgent import BasicAgent
from core.Message import UserMessage

from core.llm import EasyLLM
from skill.registry import SkillRegistry
from skill.builtin.calculator_skill import CalculatorSkill
from skill.yaml_loader import YAMLSkillLoader, MarkdownSkillLoader
from skill.folder_loader import FolderSkillLoader
from skill import MetaSkill

In [ ]:
llm= EasyLLM(provider="openai",base_url="http://127.0.0.1:5124/v1",api_key="122",model="qwen3.5-9b")

agent=BasicAgent(name="test_skill", llm=llm,reasoning={"effort":"high"},verbose_thinking=True)
agent.with_skill(CalculatorSkill())
print(llm.model)

In [ ]:
# llm.invoke_raw([UserMessage("你是?")])
agent.invoke("请仔细思考,你是?")

In [ ]:
agent.get_history()

In [ ]:
await agent.astream_invoke("你是?")

In [3]:
#自定义skill
from pydantic import BaseModel,Field
from Tool import Tool
from skill import BaseSkill
from skill import SkillConfig
class TranslateParams(BaseModel):
    text: str = Field(description="要翻译的文本")
    target_lang: str = Field(default="en", description="目标语言")

class TranslateTool(Tool):
    def __init__(self):
        super().__init__("translate_tool", "将文本翻译为目标语言", TranslateParams)

    def run(self, parameters: dict) -> str:
        # 实际翻译逻辑
        return f"Translated: {parameters['text']}"

# 2. 定义 Skill
class TranslateSkill(BaseSkill):
    def __init__(self):
        config = SkillConfig(
            name="translate",
            description="多语言翻译技能",
            version="1.0.0",
            tags=["translate", "language", "i18n"],
            priority=5,
        )
        super().__init__(config)

    def get_tools(self) -> list:
        return [TranslateTool()]

    def get_prompt(self) -> str:
        return """## 翻译能力
你具备多语言翻译能力。当用户要求翻译时，请使用 translate_tool 工具。
- 支持中英日韩等多种语言
- 可以自动识别源语言
"""


In [ ]:
agent.with_skill(TranslateSkill())


In [ ]:
from core import enable_logging
enable_logging()
agent.clear_history()
# agent._build_start_messages(f"使用工具翻译下面的文字到英语并判断这个工具正确吗:\n你是谁，在哪里 \n 并帮我计算3^22")
await agent.ainvoke(f"使用工具翻译下面的文字到英语并判断这个工具正确吗:\n你是谁，在哪里 \n 并帮我计算3^22" )

In [ ]:
await agent.astream_invoke("我们刚才说了什么")

In [ ]:
agent.get_history()

In [ ]:
message=agent._build_start_messages("111")
agent.llm._convert_messages(message)

In [ ]:
print(agent.get_enhanced_prompt())

In [ ]:
agent.get_trace_history()

In [ ]:
agent.save_session("test_00001")

In [ ]:
agent2=BasicAgent.load_session("test_00001",llm=agent.llm)

In [ ]:
from skill import SkillManager


agent_resume:BasicAgent=BasicAgent.load_session("test_00001",llm=agent.llm,tool_registry=agent.tool_registry,skill_manager=agent.skill_manager)

In [ ]:
await agent_resume.astream_invoke("我们刚才聊了什么")

In [ ]:
agent_resume.get_trace_history()

In [ ]:
manager=agent.skill_manager
prompt=manager.build_skills_prompt()
print(prompt)

In [ ]:
from skill.registry import SkillRegistry
skill_manage=SkillRegistry()
skill_manage.discover_from_directory("./real_skills/")



In [ ]:
print(skill_manage.list_available())


In [ ]:
crypto_skill=skill_manage.create('crypto_skill')
agent.with_skill(crypto_skill)
print(agent.get_enhanced_prompt())

In [ ]:
agent.invoke("i am a boy from china的 SHA-256 哈希值是什么")

In [ ]:
from memory.V2.WorkingMemory import WorkingMemory
from memory import MemoryConfig,MemoryManage
from memory.V2.Embedding.HuggingfaceEmbeddingModel import HuggingfaceEmbeddingModel
config = MemoryConfig(max_capacity=20)
working_memory = WorkingMemory(config)
mm = MemoryManage(
            config=config,
            user_id="test_integration_user",
            enable_working=True,
            working_memory=working_memory,
            enable_episodic=False,
            enable_semantic=False,
            enable_perceptual=False,
        ) 

In [ ]:
agent.with_memory(mm)
print(agent.get_enhanced_prompt())

In [ ]:
from skill.registry import SkillRegistry
from skill.builtin.calculator_skill import CalculatorSkill

# 1. 把所有 Skill 注册到全局 Registry（启动时一次性完成）
registry = SkillRegistry.instance()
registry.discover_from_directory("./real_skills/")
# 也可以从目录批量发现
# registry.discover_from_directory("./skills/")

# 2. 创建 Agent（不预加载任何 Skill）
agent1 = BasicAgent(name="assistant", llm=llm, verbose_thinking=True)
agent1.with_skill(MetaSkill(registry,manager=agent1.skill_manager))
print(agent1.get_enhanced_prompt())

In [ ]:
await agent1.astream_invoke("i am a boy from china的 SHA-256 哈希值是什么")

In [25]:
from context import ContextManager,ContextBuilder,LLMHistoryCompactor
from skill.registry import SkillRegistry
llm2= EasyLLM(provider="openai",base_url="http://127.0.0.1:5124/v1",api_key="122",model="qwen3.5-9b")

skill_manage=SkillRegistry()
skill_manage.discover_from_directory("./real_skills/")
crypto_skill=skill_manage.create('crypto_skill')

agent_context = BasicAgent(name="assistant", llm=llm2,reasoning={"effort":"high"} ,verbose_thinking=True)    
agent_context.with_skill(crypto_skill)
builder=ContextManager(max_tokens=300)
builder.set_history_compactor(LLMHistoryCompactor(llm2,recent_turns=3))
agent_context.with_context(builder)


2026-04-18 04:33:40,733 | INFO | EasyLLM 初始化完成: provider=openai, model=qwen3.5-9b
2026-04-18 04:33:40,737 | INFO | 从目录 './real_skills/' 发现并注册 1 个 Skill: ['crypto_skill']
2026-04-18 04:33:40,738 | INFO | BasicAgent 'assistant' 初始化完成，工具调用: 禁用，provider: openai
2026-04-18 04:33:40,739 | INFO | 📦 注册 Skill 'crypto_skill' (v1.0.0)
2026-04-18 04:33:40,739 | INFO | ✅ 激活 Skill 'crypto_skill' (工具: ['hash_calculator'])


In [39]:
await agent_context.astream_invoke("i am a boy from acc SHA-256 哈希值是什么")


2026-04-18 04:39:09,170 | INFO | HTTP Request: POST http://127.0.0.1:5124/v1/chat/completions "HTTP/1.1 200 OK"


round 1

thinking content:
用户再次询问同一个问题，我已经给出了答案。看起来用户可能对结果有疑虑，或者只是想确认。我应该保持简洁，重复提供相同的正确答案。

不过我注意到用户的问题是用中文问的"SHA-256 哈希值是什么"，我应该用中文回复。

让我确认一下之前的答案是否正确。我可以使用工具来验证，但既然之前已经给出了答案，而且用户只是重复提问，我应该直接确认答案。

不过为了确保准确性，我应该再次使用工具计算这个字符串的SHA-256哈希值。

content:



tool_calls:
hash_calculator : {'text': 'i am a boy from acc'}


2026-04-18 04:39:11,457 | INFO | HTTP Request: POST http://127.0.0.1:5124/v1/chat/completions "HTTP/1.1 200 OK"


  [Tool执行] 计算文本 'i am a boy from acc' 的 SHA-256 结果为: d57fd35f5c206cd42192f4930bf8abe84cd171e7fd7a9a3917324a45421ce28e

round 2

thinking content:
用户再次询问了相同的问题。我已经通过工具确认了哈希值。

字符串 "i am a boy from acc" 的 SHA-256 哈希值是：

`d57fd35f5c206cd42192f4930bf8abe84cd171e7fd7a9a3917324a45421ce28e`

我应该直接给出答案，不需要额外解释。

content:


字符串 "i am a boy from acc" 的 SHA-256 哈希值是：

`d57fd35f5c206cd42192f4930bf8abe84cd171e7fd7a9a3917324a45421ce28e`
final res:


字符串 "i am a boy from acc" 的 SHA-256 哈希值是：

`d57fd35f5c206cd42192f4930bf8abe84cd171e7fd7a9a3917324a45421ce28e`


'\n\n字符串 "i am a boy from acc" 的 SHA-256 哈希值是：\n\n`d57fd35f5c206cd42192f4930bf8abe84cd171e7fd7a9a3917324a45421ce28e`'

In [42]:
agent_context.get_context_usage()

{'label': 'astream_invoke_tool',
 'request_tokens': 2762,
 'used_tokens': 2762,
 'remaining_tokens': -2462,
 'overflow_tokens': 2462,
 'max_tokens': 300,
 'request_compacted': False,
 'request_compaction_possible': False,
 'request_tokens_before_compaction': 2762,
 'request_tokens_after_compaction': 2762,
 'overflow_tokens_before_compaction': 2462,
 'overflow_tokens_after_compaction': 2462,
 'tracked_at': '2026-04-18T04:39:11.437192'}

In [40]:
len(agent_context.get_canonical_history())

8

In [41]:
cm=LLMHistoryCompactor(llm2,recent_turns=0)
re=cm.compact(agent_context.get_canonical_history(),max_tokens=300)

2026-04-18 04:43:03,194 | INFO | Compact History


KeyboardInterrupt: 